# Medical Knowledge Base: Scraper & NLP Pipeline

This notebook acts as the main tutorial to understand the **Scraper Service** architecture we built for Certus Diagnostics. It demonstrates how we fetch medical data, extract the main content, convert it to clean Markdown, and then applies the NLP text-cleaning architecture (Corpus & Document-Term Matrix) required for the assignment.

## 1. Understanding the Scraper Service

Instead of a simple scraping script, we built a modular pipeline. Let's import the individual components to see how a document goes from a URL to clean Markdown.

In [1]:
import os
import sys
from pprint import pprint

# Ensure the root directory is in the path to import our scraper package
sys.path.append(os.path.abspath('..'))

from scraper.downloader import download_html
from scraper.extractor import extract_main_content
from scraper.markdown_converter import convert_to_markdown
from scraper.quality_checker import check_quality

### Step 1: Downloading Raw HTML
The `downloader` module handles HTTP requests, retries, and captures metadata like ETags for versioning.

In [2]:
# Let's fetch a direct article about the A1C test from NIDDK
url = "https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test"
download_result = download_html(url)

print(f"Status: Downloaded successfully. Length: {len(download_result.html)} characters.")
print(f"ETag: {download_result.etag}")

raw_html = download_result.html

Status: Downloaded successfully. Length: 59645 characters.
ETag: None


Let's look at a snippet of the **Raw HTML**. Notice how messy it is with navigation, scripts, and footers.

In [3]:
# Show the first 1000 characters of raw HTML
print(raw_html[:1000])

<!doctype html>

<html lang="en-US">

<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta http-equiv="X-UA-Compatible" content="IE=edge">
    <title>The A1C Test & Diabetes - NIDDK</title>
    
    <meta name="msvalidate.01" content="7FEF9D64C9916EC9393EB5B21BD0037E">
    
    <meta name="format-detection" content="telephone=no">
    <link rel="icon" href="/_ux/niddk/9.40.0/favicon.ico">
    <link rel="apple-touch-icon" href="/_ux/niddk/9.40.0/images/apple-touch-icon.png">
    
    <link rel="preconnect" href="https://www.google-analytics.com">
    <link rel="preconnect" href="https://www.googletagmanager.com">
    <link rel="dns-prefetch" href="https://www.google-analytics.com">
    <link rel="dns-prefetch" href="https://www.googletagmanager.com">
    <link rel="dns-prefetch" href="https://dap.digitalgov.gov">
    <link rel="dns-prefetch" href="https://livechat.niddk.nih.gov">
    <link rel="prelo


### Step 2: Content Extraction
The `extractor` module uses `trafilatura` and `BeautifulSoup` to strip away the ads, navigation, and footers, leaving only the clinical article content in a structured XML format (preserving tables and headings).

In [4]:
extracted_xml = extract_main_content(raw_html)
print(f"Extracted content length: {len(extracted_xml)} characters.")
print("\nSnippet of extracted XML:\n", extracted_xml[:1000])

Extracted content length: 18617 characters.

Snippet of extracted XML:
 <doc fingerprint="1e79ce15530f57a5">
  <main>
    <head rend="h1">The A1C Test &amp; Diabetes</head>
    <p>On this page:</p>
    <head rend="h2">What is the A1C test?</head>
    <p>The A1C test is a blood test that provides information about your average levels of blood glucose, also called blood sugar, over the past 3 months. The A1C test can be used to diagnose <ref target="/health-information/diabetes/overview/what-is-diabetes/type-2-diabetes">type 2 diabetes</ref> and <ref target="/health-information/diabetes/overview/what-is-diabetes/prediabetes-insulin-resistance">prediabetes</ref>.1 The A1C test is also the primary test used for <ref target="/health-information/diabetes/overview/managing-diabetes">diabetes management</ref>.</p>
    <graphic src="/-/media/Images/Health-Information/Diabetes/HCP_drawingblood_330x220.png" alt="A health care professional draws blood."/>
    <p>The A1C test is sometimes called th

### Step 3: Markdown Conversion
The `markdown_converter` takes that structured XML and turns it into clean, readable Markdown suitable for our Knowledge Base and Vector DB.

In [5]:
clean_markdown = convert_to_markdown(extracted_xml, metadata_header={"url": url, "source": "NIDDK"})
print("=== CLEANED MARKDOWN ===\n")
print(clean_markdown[:1500])

=== CLEANED MARKDOWN ===

---
url: https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test
source: NIDDK
---

The A1C Test & Diabetes

On this page:

What is the A1C test?

The A1C test is a blood test that provides information about your average levels of blood glucose, also called blood sugar, over the past 3 months. The A1C test can be used to diagnose type 2 diabetes and prediabetes.1 The A1C test is also the primary test used for diabetes management.

The A1C test is sometimes called the hemoglobin A1C, HbA1c, glycated hemoglobin, or glycohemoglobin test. Hemoglobin is the part of a red blood cell that carries oxygen to the cells. Glucose attaches to or binds with hemoglobin in your blood cells, and the A1C test is based on this attachment of glucose to hemoglobin.

The higher the glucose level in your bloodstream, the more glucose will attach to the hemoglobin. The A1C test measures the amount of hemoglobin with attached glucose and reflects your average blood gluc

### Step 4: Quality Checker
Before we save this to our corpus, the `quality_checker` ensures it's not garbage data (e.g., a 404 page or a search hub).

In [6]:
quality_result = check_quality(clean_markdown, keywords=["a1c", "diabetes", "blood"])
pprint(quality_result.__dict__)

{'has_headings': False,
 'has_keywords': True,
 'is_english': True,
 'min_words': True,
 'passed': False,
 'reasons': ['No markdown headings found'],
 'word_count': 2163}


---

## 2. NLP Assignment 2 (Corpus & DTM)

Now that we understand how the scraper produces high-quality text, let's load a few scraped documents (or simulate them here) and apply the standard Data Cleaning and NLP structuring techniques: **Corpus** and **Document-Term Matrix**.

In [7]:
# Let's fetch a few more articles to build our corpus
urls = {
    'a1c': 'https://www.niddk.nih.gov/health-information/diagnostic-tests/a1c-test',
    'insulin_resistance': 'https://www.niddk.nih.gov/health-information/diabetes/overview/what-is-diabetes/prediabetes-insulin-resistance',
    'kidney_tests': 'https://www.niddk.nih.gov/health-information/kidney-disease/chronic-kidney-disease-ckd/tests-diagnosis'
}

data = {}
for topic, u in urls.items():
    print(f"Fetching {topic}...")
    html = download_html(u).html
    xml = extract_main_content(html)
    md = convert_to_markdown(xml)
    data[topic] = [md] # Store as a list of strings to match the assignment format
    
print("Data loaded successfully!")

Fetching a1c...
Fetching insulin_resistance...
Fetching kidney_tests...
Data loaded successfully!


### Data Cleaning
We will remove punctuation, lowercase the text, and remove numbers to prepare it for tokenization.

In [8]:
import pandas as pd
import re
import string

pd.set_option('max_colwidth', 150)

# Create DataFrame
data_df = pd.DataFrame.from_dict(data).transpose()
data_df.columns = ['transcript']
data_df = data_df.sort_index()

# Round 1 Cleaning
def clean_text_round1(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\w*\d\w*', '', text)
    return text

# Round 2 Cleaning (New lines)
def clean_text_round2(text):
    text = re.sub('[‘’“”…]', '', text)
    text = re.sub('\n', ' ', text)
    return text

data_clean = pd.DataFrame(data_df.transcript.apply(lambda x: clean_text_round1(x)))
data_clean = pd.DataFrame(data_clean.transcript.apply(lambda x: clean_text_round2(x)))

# The Corpus
data_clean['full_name'] = ['A1C Test', 'Insulin Resistance', 'Kidney Tests']
data_clean

<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
<>:15: SyntaxWarning: invalid escape sequence '\['
<>:17: SyntaxWarning: invalid escape sequence '\w'
/var/folders/83/308cnjqn2dz8h3lcgy6q00w00000gn/T/ipykernel_89897/504505893.py:15: SyntaxWarning: invalid escape sequence '\['
  text = re.sub('\[.*?\]', '', text)
/var/folders/83/308cnjqn2dz8h3lcgy6q00w00000gn/T/ipykernel_89897/504505893.py:17: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub('\w*\d\w*', '', text)


,transcript,full_name
a1c,the test diabetes on this page what is the test the test is a blood test that provides information about your average levels of blood gluco...,A1C Test
insulin_resistance,insulin resistance prediabetes on this page what are insulin resistance and prediabetes insulin resistance is a condition in which your body d...,Insulin Resistance
kidney_tests,chronic kidney disease tests diagnosis how can i tell if i have kidney disease early kidney disease usually doesnt have any symptoms testing is ...,Kidney Tests


### Document-Term Matrix (DTM)
Using `CountVectorizer`, we tokenize the text, remove stop words, and create a matrix of word frequencies.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(stop_words='english')
data_cv = cv.fit_transform(data_clean.transcript)
data_dtm = pd.DataFrame(data_cv.toarray(), columns=cv.get_feature_names_out())
data_dtm.index = data_clean.index

data_dtm.head()

,able,acromegaly,active,activemanaging,activesmoking,activewhat,activity,adults,adultshow,advanced,...,worked,working,worldwide,worried,worse,year,years,yes,younger,youre
a1c,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,2,2,0,1
insulin_resistance,2,1,1,1,1,1,1,2,1,0,...,1,1,1,1,0,0,2,0,1,3
kidney_tests,0,0,0,0,0,0,0,0,0,0,...,0,4,0,0,2,1,0,0,0,0


### Saving the Data
Finally, we pickle the corpus, the DTM, and the Vectorizer for future assignments.

In [10]:
import pickle

# Save Corpus
data_clean.to_pickle("corpus.pkl")

# Save Document-Term Matrix
data_dtm.to_pickle("dtm.pkl")

# Save CountVectorizer
pickle.dump(cv, open("cv.pkl", "wb"))

print("NLP artifacts saved successfully!")

NLP artifacts saved successfully!
